# 03 SQL Database and Analysis Views

## Project: PG&E-Style Utility Operations & Meter-to-Cash Analytics

This notebook creates the SQL database layer for the project. The goal is to load the processed real-data tables and synthetic Meter-to-Cash tables into a SQLite database, validate the loaded tables, and create SQL views for operational reporting and Power BI dashboard development.

Notebook 1 created the real-data foundation using California outage data and CEC electricity consumption data. Notebook 2 created the synthetic Meter-to-Cash layer using real PG&E consumption patterns to guide customer, meter-read, billing, exception, and service request generation.

In this notebook, I will:

1. Load processed real-data tables and synthetic Meter-to-Cash tables from CSV files.
2. Create a SQLite database for the project.
3. Write the dataframes into SQL tables.
4. Validate row counts and key relationships in SQL.
5. Create SQL views for Meter-to-Cash KPIs, billing exceptions, bill status reporting, service request backlog, account readiness, monthly billing trends, and PG&E outage/consumption context.
6. Export analysis-ready SQL outputs for Power BI dashboard development.

The purpose of this notebook is to create a reusable relational database layer that connects the project’s real-data foundation with the synthetic enterprise billing workflow.

## 1. Import Libraries and Define Project Paths

Because this notebook starts from a fresh Jupyter session, all required libraries, paths, and input files are reloaded from disk. The processed and synthetic CSV outputs from the first two notebooks are used as the source data for the SQL database.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

# Define project paths
PROCESSED_DIR = Path("../data/processed")
SYNTHETIC_DIR = Path("../data/synthetic")
DATABASE_DIR = Path("../data/database")

# Create database directory if it does not exist
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

# Define SQLite database path
DB_PATH = DATABASE_DIR / "pge_meter_to_cash.db"

# Confirm available input files
processed_files = sorted([file.name for file in PROCESSED_DIR.iterdir()])
synthetic_files = sorted([file.name for file in SYNTHETIC_DIR.iterdir()])

print("Processed files:")
display(processed_files)

print("Synthetic files:")
display(synthetic_files)

Processed files:


['cec_county_monthly_summary.csv',
 'county_consumption_2024.csv',
 'outage_county_clean.csv',
 'outage_incidents_clean.csv',
 'pge_cause_category_summary.csv',
 'pge_cause_summary.csv',
 'pge_county_normalized_impact.csv',
 'pge_county_outage_consumption_context.csv',
 'pge_county_outage_summary.csv',
 'pge_monthly_consumption_2024.csv',
 'pge_outage_incidents.csv',
 'pge_outage_type_summary.csv',
 'pge_utility_sector_2024.csv',
 'utility_consumption_2024.csv',
 'utility_outage_summary.csv']

Synthetic files:


['billing_cycles.csv',
 'billing_exceptions.csv',
 'bills.csv',
 'customers.csv',
 'meter_reads.csv',
 'meters.csv',
 'monthly_usage_multipliers.csv',
 'rate_plans.csv',
 'relationship_validation.csv',
 'service_accounts.csv',
 'service_requests.csv',
 'uat_test_cases.csv']

### Input File Inventory Validation

The processed and synthetic input files loaded successfully from disk.

Key observations:

- The processed folder contains the real-data outputs from Notebook 1, including outage summaries, PG&E outage context, county consumption summaries, and PG&E utility consumption summaries.
- The synthetic folder contains the Meter-to-Cash outputs from Notebook 2, including customers, service accounts, meters, meter reads, bills, billing exceptions, service requests, rate plans, billing cycles, and UAT test cases.
- Because all required CSV files are available, this notebook can build the SQLite database from disk without rerunning the earlier notebooks.

## 2. Load Processed and Synthetic CSV Files

This section loads the processed real-data tables and synthetic Meter-to-Cash tables into pandas dataframes before writing them to SQLite.

In [2]:
# Load processed real-data tables from Notebook 1
processed_tables = {
    "cec_county_monthly_summary": pd.read_csv(PROCESSED_DIR / "cec_county_monthly_summary.csv"),
    "county_consumption_2024": pd.read_csv(PROCESSED_DIR / "county_consumption_2024.csv"),
    "outage_county_clean": pd.read_csv(PROCESSED_DIR / "outage_county_clean.csv"),
    "outage_incidents_clean": pd.read_csv(PROCESSED_DIR / "outage_incidents_clean.csv"),
    "pge_cause_category_summary": pd.read_csv(PROCESSED_DIR / "pge_cause_category_summary.csv"),
    "pge_cause_summary": pd.read_csv(PROCESSED_DIR / "pge_cause_summary.csv"),
    "pge_county_normalized_impact": pd.read_csv(PROCESSED_DIR / "pge_county_normalized_impact.csv"),
    "pge_county_outage_consumption_context": pd.read_csv(PROCESSED_DIR / "pge_county_outage_consumption_context.csv"),
    "pge_county_outage_summary": pd.read_csv(PROCESSED_DIR / "pge_county_outage_summary.csv"),
    "pge_monthly_consumption_2024": pd.read_csv(PROCESSED_DIR / "pge_monthly_consumption_2024.csv"),
    "pge_outage_incidents": pd.read_csv(PROCESSED_DIR / "pge_outage_incidents.csv"),
    "pge_outage_type_summary": pd.read_csv(PROCESSED_DIR / "pge_outage_type_summary.csv"),
    "pge_utility_sector_2024": pd.read_csv(PROCESSED_DIR / "pge_utility_sector_2024.csv"),
    "utility_consumption_2024": pd.read_csv(PROCESSED_DIR / "utility_consumption_2024.csv"),
    "utility_outage_summary": pd.read_csv(PROCESSED_DIR / "utility_outage_summary.csv")
}

# Load synthetic Meter-to-Cash tables from Notebook 2
synthetic_tables = {
    "billing_cycles": pd.read_csv(SYNTHETIC_DIR / "billing_cycles.csv"),
    "billing_exceptions": pd.read_csv(SYNTHETIC_DIR / "billing_exceptions.csv"),
    "bills": pd.read_csv(SYNTHETIC_DIR / "bills.csv"),
    "customers": pd.read_csv(SYNTHETIC_DIR / "customers.csv"),
    "meter_reads": pd.read_csv(SYNTHETIC_DIR / "meter_reads.csv"),
    "meters": pd.read_csv(SYNTHETIC_DIR / "meters.csv"),
    "monthly_usage_multipliers": pd.read_csv(SYNTHETIC_DIR / "monthly_usage_multipliers.csv"),
    "rate_plans": pd.read_csv(SYNTHETIC_DIR / "rate_plans.csv"),
    "relationship_validation": pd.read_csv(SYNTHETIC_DIR / "relationship_validation.csv"),
    "service_accounts": pd.read_csv(SYNTHETIC_DIR / "service_accounts.csv"),
    "service_requests": pd.read_csv(SYNTHETIC_DIR / "service_requests.csv"),
    "uat_test_cases": pd.read_csv(SYNTHETIC_DIR / "uat_test_cases.csv")
}

print("Processed tables loaded:", len(processed_tables))
print("Synthetic tables loaded:", len(synthetic_tables))

Processed tables loaded: 15
Synthetic tables loaded: 12


### Table Load Validation

All expected processed and synthetic tables loaded successfully.

Key observations:

- 15 processed real-data tables were loaded from Notebook 1 outputs.
- 12 synthetic Meter-to-Cash tables were loaded from Notebook 2 outputs.
- The project now has both the real utility data foundation and the synthetic enterprise billing layer available for SQL database creation.

## 3. Review Table Dimensions Before Database Load

Before writing the dataframes to SQLite, this section reviews table dimensions to confirm that each dataframe loaded with the expected number of rows and columns.

In [3]:
# Create table inventory for processed tables
processed_inventory = pd.DataFrame([
    {
        "table_group": "processed",
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in processed_tables.items()
])

# Create table inventory for synthetic tables
synthetic_inventory = pd.DataFrame([
    {
        "table_group": "synthetic",
        "table_name": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in synthetic_tables.items()
])

table_inventory = pd.concat(
    [processed_inventory, synthetic_inventory],
    ignore_index=True
).sort_values(["table_group", "table_name"])

table_inventory

,table_group,table_name,rows,columns
0,processed,cec_county_monthly_summary,11832,7
1,processed,county_consumption_2024,58,5
2,processed,outage_county_clean,47,11
3,processed,outage_incidents_clean,258,19
4,processed,pge_cause_category_summary,8,6
5,processed,pge_cause_summary,13,6
6,processed,pge_county_normalized_impact,37,7
7,processed,pge_county_outage_consumption_context,37,13
8,processed,pge_county_outage_summary,37,7
9,processed,pge_monthly_consumption_2024,12,6


### Table Inventory Validation

The table inventory confirms that the processed and synthetic dataframes loaded with the expected dimensions.

Key observations:

- The synthetic Meter-to-Cash layer includes 5,000 customers, service accounts, and meters.
- The usage and billing layers each contain 60,000 records, representing 5,000 accounts across 12 monthly billing cycles.
- The billing exception and service request tables contain operational workflow records for exception monitoring and follow-up.
- The processed real-data layer includes PG&E outage summaries, electricity consumption context, and PG&E monthly/sector consumption tables from Notebook 1.
- The table sizes are consistent with the design decisions from the first two notebooks and are ready to be written to SQLite.

## 4. Write Tables to SQLite Database

This section creates the SQLite database and writes all processed and synthetic dataframes as database tables. Existing tables are replaced so the notebook can be rerun cleanly.

In [4]:
# Connect to SQLite database
conn = sqlite3.connect(DB_PATH)

# Write processed tables to SQLite
for table_name, df in processed_tables.items():
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

# Write synthetic tables to SQLite
for table_name, df in synthetic_tables.items():
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

# Confirm tables written to database
sqlite_tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

sqlite_tables

,name
0,billing_cycles
1,billing_exceptions
2,bills
3,cec_county_monthly_summary
4,county_consumption_2024
5,customers
6,meter_reads
7,meters
8,monthly_usage_multipliers
9,outage_county_clean


### SQLite Table Creation Validation

The SQLite database was created successfully and all expected tables were written to the database.

Key observations:

- The database contains the processed real-data tables from Notebook 1.
- The database contains the synthetic Meter-to-Cash tables from Notebook 2.
- The table list includes outage, consumption, customer, account, meter, billing, exception, service request, rate plan, billing cycle, and UAT tables.
- This confirms that the project now has a reusable relational database layer for SQL analysis and reporting.

The next step is to validate row counts directly from SQLite to confirm that the database tables match the source dataframe inventory.

## 5. Validate SQLite Row Counts

This section validates that each SQLite table contains the expected number of rows after being written to the database.

In [7]:
# Validate row counts from SQLite
sqlite_row_counts = []

for table_name in sqlite_tables["name"]:
    row_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {table_name};",
        conn
    )["row_count"].iloc[0]
    
    sqlite_row_counts.append({
        "table_name": table_name,
        "sqlite_row_count": row_count
    })

sqlite_row_counts = pd.DataFrame(sqlite_row_counts)

# Join SQLite row counts to dataframe inventory
row_count_validation = table_inventory.merge(
    sqlite_row_counts,
    on="table_name",
    how="left"
)

row_count_validation["row_count_match"] = (
    row_count_validation["rows"] == row_count_validation["sqlite_row_count"]
)

row_count_validation.sort_values("table_name")

,table_group,table_name,rows,columns,sqlite_row_count,row_count_match
15,synthetic,billing_cycles,12,5,12,True
16,synthetic,billing_exceptions,5403,12,5403,True
17,synthetic,bills,60000,13,60000,True
0,processed,cec_county_monthly_summary,11832,7,11832,True
1,processed,county_consumption_2024,58,5,58,True
18,synthetic,customers,5000,5,5000,True
19,synthetic,meter_reads,60000,9,60000,True
20,synthetic,meters,5000,5,5000,True
21,synthetic,monthly_usage_multipliers,12,4,12,True
2,processed,outage_county_clean,47,11,47,True


### SQLite Row Count Validation Findings

The SQLite row count validation confirms that all tables were written to the database correctly.

Key observations:

- Every SQLite table has a row count that matches the original dataframe row count.
- No missing or partially loaded tables were identified.
- Large tables such as `meter_reads` and `bills` loaded successfully with 60,000 records each.
- Core dimension tables such as `customers`, `service_accounts`, `meters`, `rate_plans`, and `billing_cycles` loaded with the expected counts.
- Processed real-data tables from Notebook 1 also loaded with matching row counts.

This validation confirms that the SQLite database is ready for SQL relationship checks and analysis view creation.

## 6. Validate Key Relationships in SQL

This section validates the main relational links in SQLite to confirm that the database does not contain orphan records across the core Meter-to-Cash tables.

In [6]:
relationship_queries = {
    "service_accounts_without_customer": """
        SELECT COUNT(*) AS orphan_record_count
        FROM service_accounts sa
        LEFT JOIN customers c ON sa.customer_id = c.customer_id
        WHERE c.customer_id IS NULL;
    """,
    "meters_without_account": """
        SELECT COUNT(*) AS orphan_record_count
        FROM meters m
        LEFT JOIN service_accounts sa ON m.account_id = sa.account_id
        WHERE sa.account_id IS NULL;
    """,
    "meter_reads_without_account": """
        SELECT COUNT(*) AS orphan_record_count
        FROM meter_reads mr
        LEFT JOIN service_accounts sa ON mr.account_id = sa.account_id
        WHERE sa.account_id IS NULL;
    """,
    "bills_without_account": """
        SELECT COUNT(*) AS orphan_record_count
        FROM bills b
        LEFT JOIN service_accounts sa ON b.account_id = sa.account_id
        WHERE sa.account_id IS NULL;
    """,
    "exceptions_without_bill": """
        SELECT COUNT(*) AS orphan_record_count
        FROM billing_exceptions be
        LEFT JOIN bills b ON be.bill_id = b.bill_id
        WHERE b.bill_id IS NULL;
    """,
    "service_requests_without_account": """
        SELECT COUNT(*) AS orphan_record_count
        FROM service_requests sr
        LEFT JOIN service_accounts sa ON sr.account_id = sa.account_id
        WHERE sa.account_id IS NULL;
    """
}

sql_relationship_validation = []

for check_name, query in relationship_queries.items():
    result = pd.read_sql_query(query, conn)["orphan_record_count"].iloc[0]
    sql_relationship_validation.append({
        "relationship_check": check_name,
        "orphan_record_count": result
    })

sql_relationship_validation = pd.DataFrame(sql_relationship_validation)
sql_relationship_validation

,relationship_check,orphan_record_count
0,service_accounts_without_customer,0
1,meters_without_account,0
2,meter_reads_without_account,0
3,bills_without_account,0
4,exceptions_without_bill,0
5,service_requests_without_account,0


### SQL Relationship Validation Findings

The SQL relationship checks confirm that the core database tables are connected correctly.

No orphan records were found across the main customer, account, meter, billing, exception, and service request relationships. This confirms that the SQLite database is ready for analysis view creation.

## 7. Create Meter-to-Cash KPI View

This section creates a SQL view that summarizes high-level Meter-to-Cash performance by billing cycle. The view combines bill volume, bill status, billed usage, billed revenue, exceptions, and service request activity into one monthly KPI table.

In [8]:
# Create Meter-to-Cash monthly KPI view
create_meter_to_cash_kpi_view = """
DROP VIEW IF EXISTS vw_meter_to_cash_monthly_kpis;

CREATE VIEW vw_meter_to_cash_monthly_kpis AS
SELECT
    b.billing_cycle_id,
    COUNT(DISTINCT b.account_id) AS accounts_billed,
    COUNT(DISTINCT b.bill_id) AS total_bills,
    
    SUM(CASE WHEN b.bill_status = 'Paid' THEN 1 ELSE 0 END) AS paid_bills,
    SUM(CASE WHEN b.bill_status = 'Generated' THEN 1 ELSE 0 END) AS generated_bills,
    SUM(CASE WHEN b.bill_status = 'Past Due' THEN 1 ELSE 0 END) AS past_due_bills,
    SUM(CASE WHEN b.bill_status = 'Exception' THEN 1 ELSE 0 END) AS exception_bills,
    
    ROUND(SUM(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount ELSE 0 END), 2) AS total_bill_amount,
    ROUND(AVG(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount END), 2) AS avg_bill_amount,
    ROUND(SUM(CASE WHEN b.kwh_usage >= 0 THEN b.kwh_usage ELSE 0 END), 2) AS total_kwh_usage,
    
    COUNT(DISTINCT be.exception_id) AS billing_exceptions,
    SUM(CASE WHEN be.resolution_status = 'Open' THEN 1 ELSE 0 END) AS open_exceptions,
    SUM(CASE WHEN be.resolution_status = 'In Review' THEN 1 ELSE 0 END) AS in_review_exceptions,
    SUM(CASE WHEN be.resolution_status = 'Resolved' THEN 1 ELSE 0 END) AS resolved_exceptions,
    
    COUNT(DISTINCT sr.request_id) AS service_requests,
    SUM(CASE WHEN sr.request_status = 'Open' THEN 1 ELSE 0 END) AS open_service_requests,
    SUM(CASE WHEN sr.request_status = 'In Progress' THEN 1 ELSE 0 END) AS in_progress_service_requests,
    SUM(CASE WHEN sr.request_status = 'Closed' THEN 1 ELSE 0 END) AS closed_service_requests,
    
    ROUND(
        100.0 * SUM(CASE WHEN b.bill_status = 'Exception' THEN 1 ELSE 0 END) 
        / COUNT(DISTINCT b.bill_id),
        2
    ) AS bill_exception_rate_pct,
    
    ROUND(
        100.0 * SUM(CASE WHEN b.bill_status = 'Past Due' THEN 1 ELSE 0 END) 
        / COUNT(DISTINCT b.bill_id),
        2
    ) AS past_due_rate_pct

FROM bills b
LEFT JOIN billing_exceptions be
    ON b.bill_id = be.bill_id
LEFT JOIN service_requests sr
    ON b.bill_id = sr.bill_id
GROUP BY
    b.billing_cycle_id;
"""

conn.executescript(create_meter_to_cash_kpi_view)

# Preview the new view
meter_to_cash_kpis = pd.read_sql_query(
    """
    SELECT *
    FROM vw_meter_to_cash_monthly_kpis
    ORDER BY billing_cycle_id;
    """,
    conn
)

meter_to_cash_kpis

,billing_cycle_id,accounts_billed,total_bills,paid_bills,generated_bills,past_due_bills,exception_bills,total_bill_amount,avg_bill_amount,total_kwh_usage,billing_exceptions,open_exceptions,in_review_exceptions,resolved_exceptions,service_requests,open_service_requests,in_progress_service_requests,closed_service_requests,bill_exception_rate_pct,past_due_rate_pct
0,2024-01,5000,5000,3271,888,395,446,3486346.40,765.56,15364540.59,446,132,60,254,204,47,38,112,8.92,7.90
1,2024-02,5000,5000,3284,890,375,451,3019801.79,663.84,13477209.44,451,138,70,243,217,49,39,118,9.02,7.50
2,2024-03,5000,5000,3289,902,366,443,2967342.36,651.16,13355506.15,443,135,66,242,189,40,23,116,8.86,7.32
3,2024-04,5000,5000,3256,906,390,448,2800526.57,615.23,12678500.02,448,139,68,241,198,49,27,110,8.96,7.80
4,2024-05,5000,5000,3298,895,350,457,2849878.10,627.31,12908880.53,457,131,70,256,201,38,23,124,9.14,7.00
5,2024-06,5000,5000,3243,948,340,469,3177537.96,701.29,14223252.82,469,144,80,245,220,42,43,122,9.38,6.80
6,2024-07,5000,5000,3223,971,364,442,4432771.25,972.53,19095483.96,442,127,71,244,216,37,38,123,8.84,7.28
7,2024-08,5000,5000,3234,975,347,444,4376210.22,960.54,18920978.99,444,118,74,252,200,46,28,118,8.88,6.94
8,2024-09,5000,5000,3196,976,372,456,3671254.90,807.93,16116547.28,456,141,71,244,206,46,38,110,9.12,7.44
9,2024-10,5000,5000,3329,891,332,448,3820346.03,839.27,16721511.28,448,137,71,240,212,51,36,118,8.96,6.64


### Meter-to-Cash KPI View Findings

The `vw_meter_to_cash_monthly_kpis` view successfully summarizes monthly billing performance across the synthetic Meter-to-Cash process.

Key observations:

- Each billing cycle contains 5,000 accounts and 5,000 bills, matching the synthetic data design.
- Total bill amount and total kWh usage vary by month because meter reads were generated using real PG&E 2024 monthly usage multipliers.
- July and August show higher total usage and bill amounts, reflecting the summer demand pattern established from real PG&E consumption data.
- Exception bill counts remain relatively stable across months, creating a consistent operational backlog for exception monitoring.
- This view is useful for executive-level reporting because it combines billing volume, revenue, usage, exception activity, and service request activity into one monthly KPI table.

This SQL view will be a key source table for the Power BI dashboard.

## 8. Create Billing Exception Summary View

This section creates a SQL view for billing exception monitoring. The view summarizes exceptions by billing cycle, exception type, severity, resolution status, and customer segment.

This view will support operational dashboard metrics such as open exceptions, high-severity exceptions, resolved exceptions, and exception trends over time.

In [9]:
# Create billing exception summary view
create_billing_exception_summary_view = """
DROP VIEW IF EXISTS vw_billing_exception_summary;

CREATE VIEW vw_billing_exception_summary AS
SELECT
    be.billing_cycle_id,
    be.customer_segment,
    be.county,
    be.exception_type,
    be.severity,
    be.resolution_status,
    COUNT(DISTINCT be.exception_id) AS exception_count,
    COUNT(DISTINCT be.account_id) AS affected_accounts,
    
    SUM(CASE WHEN be.resolution_status = 'Open' THEN 1 ELSE 0 END) AS open_exceptions,
    SUM(CASE WHEN be.resolution_status = 'In Review' THEN 1 ELSE 0 END) AS in_review_exceptions,
    SUM(CASE WHEN be.resolution_status = 'Resolved' THEN 1 ELSE 0 END) AS resolved_exceptions,
    
    SUM(CASE WHEN be.severity = 'High' THEN 1 ELSE 0 END) AS high_severity_exceptions,
    SUM(CASE WHEN be.severity = 'Medium' THEN 1 ELSE 0 END) AS medium_severity_exceptions

FROM billing_exceptions be
GROUP BY
    be.billing_cycle_id,
    be.customer_segment,
    be.county,
    be.exception_type,
    be.severity,
    be.resolution_status;
"""

conn.executescript(create_billing_exception_summary_view)

# Preview exception summary view
billing_exception_summary = pd.read_sql_query(
    """
    SELECT *
    FROM vw_billing_exception_summary
    ORDER BY billing_cycle_id, exception_count DESC
    LIMIT 20;
    """,
    conn
)

billing_exception_summary

,billing_cycle_id,customer_segment,county,exception_type,severity,resolution_status,exception_count,affected_accounts,open_exceptions,in_review_exceptions,resolved_exceptions,high_severity_exceptions,medium_severity_exceptions
0,2024-01,Residential,KERN,Inactive Account,High,Resolved,19,19,0,0,19,19,0
1,2024-01,Residential,SANTA CLARA,Inactive Account,High,Resolved,18,18,0,0,18,18,0
2,2024-01,Residential,CONTRA COSTA,Inactive Account,High,Resolved,17,17,0,0,17,17,0
3,2024-01,Residential,ALAMEDA,Inactive Account,High,Resolved,14,14,0,0,14,14,0
4,2024-01,Residential,SANTA CLARA,Inactive Account,High,Open,14,14,14,0,0,14,0
5,2024-01,Residential,KERN,Inactive Account,High,Open,12,12,12,0,0,12,0
6,2024-01,Residential,ALAMEDA,Inactive Account,High,Open,10,10,10,0,0,10,0
7,2024-01,Residential,FRESNO,Inactive Account,High,Resolved,10,10,0,0,10,10,0
8,2024-01,Residential,PLACER,Inactive Account,High,Resolved,8,8,0,0,8,8,0
9,2024-01,Residential,SAN JOAQUIN,Inactive Account,High,Resolved,8,8,0,0,8,8,0


### Billing Exception Summary View Findings

The `vw_billing_exception_summary` view successfully creates a detailed exception reporting table by billing cycle, customer segment, county, exception type, severity, and resolution status.

Key observations:

- The view supports detailed filtering by month, geography, customer segment, exception type, severity, and status.
- Inactive account exceptions appear frequently because inactive-account billing activity was intentionally flagged as a high-severity exception.
- Missing meter read and invalid meter read exceptions are also represented, supporting data quality and meter-read monitoring use cases.
- This detailed view is better suited for Power BI slicers and drilldowns than for a simple notebook summary.

The next step is to create higher-level exception rollups for easier notebook review and dashboard KPI cards.

In [10]:
# Create a simpler rollup from the exception summary view for notebook review
exception_rollup_preview = pd.read_sql_query(
    """
    SELECT
        exception_type,
        severity,
        resolution_status,
        SUM(exception_count) AS total_exceptions,
        SUM(open_exceptions) AS open_exceptions,
        SUM(in_review_exceptions) AS in_review_exceptions,
        SUM(resolved_exceptions) AS resolved_exceptions,
        SUM(high_severity_exceptions) AS high_severity_exceptions,
        SUM(medium_severity_exceptions) AS medium_severity_exceptions
    FROM vw_billing_exception_summary
    GROUP BY
        exception_type,
        severity,
        resolution_status
    ORDER BY
        total_exceptions DESC;
    """,
    conn
)

exception_rollup_preview

,exception_type,severity,resolution_status,total_exceptions,open_exceptions,in_review_exceptions,resolved_exceptions,high_severity_exceptions,medium_severity_exceptions
0,Inactive Account,High,Resolved,2082,0,0,2082,2082,0
1,Inactive Account,High,Open,1150,1150,0,0,1150,0
2,Inactive Account,High,In Review,584,0,584,0,584,0
3,Missing Meter Read,Medium,Resolved,578,0,0,578,0,578
4,Missing Meter Read,Medium,Open,324,324,0,0,0,324
5,Invalid Meter Read,High,Resolved,280,0,0,280,280,0
6,Missing Meter Read,Medium,In Review,161,0,161,0,0,161
7,Invalid Meter Read,High,Open,149,149,0,0,149,0
8,Invalid Meter Read,High,In Review,95,0,95,0,95,0


### Billing Exception Rollup Findings

The exception rollup provides a clearer high-level view of billing exception volume by type, severity, and resolution status.

Key observations:

- Inactive account exceptions are the largest exception type and are classified as high severity.
- Most inactive account exceptions are resolved, but a meaningful number remain open or in review.
- Missing meter read exceptions are medium severity and represent the second-largest exception category.
- Invalid meter read exceptions are high severity because they indicate usage data that should not produce a normal bill.
- This rollup is useful for dashboard KPI cards and operational monitoring, while the detailed view supports drilldowns by month, county, and customer segment.

## 9. Create Bill Status Summary View

This section creates a SQL view for bill status reporting by billing cycle, customer segment, county, and rate plan. The view summarizes paid, generated, past-due, and exception bills, along with bill amounts and usage.

In [11]:
# Create bill status summary view
create_bill_status_summary_view = """
DROP VIEW IF EXISTS vw_bill_status_summary;

CREATE VIEW vw_bill_status_summary AS
SELECT
    b.billing_cycle_id,
    b.customer_segment,
    b.county,
    b.rate_plan_id,
    b.bill_status,
    
    COUNT(DISTINCT b.bill_id) AS bill_count,
    COUNT(DISTINCT b.account_id) AS account_count,
    
    ROUND(SUM(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount ELSE 0 END), 2) AS total_bill_amount,
    ROUND(AVG(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount END), 2) AS avg_bill_amount,
    ROUND(SUM(CASE WHEN b.kwh_usage >= 0 THEN b.kwh_usage ELSE 0 END), 2) AS total_kwh_usage,
    ROUND(AVG(CASE WHEN b.kwh_usage >= 0 THEN b.kwh_usage END), 2) AS avg_kwh_usage,
    
    SUM(CASE WHEN b.bill_status = 'Paid' THEN 1 ELSE 0 END) AS paid_bills,
    SUM(CASE WHEN b.bill_status = 'Generated' THEN 1 ELSE 0 END) AS generated_bills,
    SUM(CASE WHEN b.bill_status = 'Past Due' THEN 1 ELSE 0 END) AS past_due_bills,
    SUM(CASE WHEN b.bill_status = 'Exception' THEN 1 ELSE 0 END) AS exception_bills

FROM bills b
GROUP BY
    b.billing_cycle_id,
    b.customer_segment,
    b.county,
    b.rate_plan_id,
    b.bill_status;
"""

conn.executescript(create_bill_status_summary_view)

# Preview bill status summary view
bill_status_summary = pd.read_sql_query(
    """
    SELECT *
    FROM vw_bill_status_summary
    ORDER BY billing_cycle_id, total_bill_amount DESC
    LIMIT 20;
    """,
    conn
)

bill_status_summary

,billing_cycle_id,customer_segment,county,rate_plan_id,bill_status,bill_count,account_count,total_bill_amount,avg_bill_amount,total_kwh_usage,avg_kwh_usage,paid_bills,generated_bills,past_due_bills,exception_bills
0,2024-01,Industrial,ALAMEDA,R-IND-GEN,Paid,8,8,113749.85,14218.73,524917.29,65614.66,8,0,0,0
1,2024-01,Industrial,KERN,R-IND-GEN,Paid,12,12,113406.11,9450.51,546119.21,45509.93,12,0,0,0
2,2024-01,Industrial,SANTA CLARA,R-IND-GEN,Paid,9,9,108973.07,12108.12,512066.53,56896.28,9,0,0,0
3,2024-01,Residential,SANTA CLARA,R-RES-TOU,Paid,404,404,77426.91,191.65,248609.73,615.37,404,0,0,0
4,2024-01,Residential,KERN,R-RES-TOU,Paid,409,409,74746.24,182.75,240521.79,588.07,409,0,0,0
5,2024-01,Commercial,SANTA CLARA,R-COM-GEN,Paid,55,55,70847.85,1288.14,259139.00,4711.62,55,0,0,0
6,2024-01,Industrial,FRESNO,R-IND-GEN,Paid,7,7,69727.26,9961.04,335168.19,47881.17,7,0,0,0
7,2024-01,Commercial,KERN,R-COM-GEN,Paid,53,53,69446.41,1310.31,253431.96,4781.74,53,0,0,0
8,2024-01,Commercial,ALAMEDA,R-COM-GEN,Paid,43,43,64852.22,1508.19,233016.63,5418.99,43,0,0,0
9,2024-01,"Transportation, Communications, & Utilities",SANTA CLARA,R-TCU-GEN,Paid,13,13,61769.54,4751.50,245881.57,18913.97,13,0,0,0


### Bill Status Summary View Findings

The `vw_bill_status_summary` view successfully summarizes billing outcomes by month, customer segment, county, rate plan, and bill status.

Key observations:

- The view supports detailed bill status reporting across paid, generated, past-due, and exception bills.
- Bill amounts and kWh usage are summarized by customer segment, county, and rate plan.
- Industrial and commercial records appear near the top of the preview because they have higher usage and bill amounts than residential accounts.
- This detailed view is useful for Power BI slicers and drilldowns by billing cycle, geography, customer segment, rate plan, and bill status.

The next step is to create a simpler rollup that summarizes bill status across the full billing dataset.

In [12]:
# Create a simpler bill status rollup for notebook review
bill_status_rollup = pd.read_sql_query(
    """
    SELECT
        bill_status,
        COUNT(*) AS bill_count,
        COUNT(DISTINCT account_id) AS affected_accounts,
        ROUND(SUM(CASE WHEN bill_amount IS NOT NULL THEN bill_amount ELSE 0 END), 2) AS total_bill_amount,
        ROUND(AVG(CASE WHEN bill_amount IS NOT NULL THEN bill_amount END), 2) AS avg_bill_amount,
        ROUND(SUM(CASE WHEN kwh_usage >= 0 THEN kwh_usage ELSE 0 END), 2) AS total_kwh_usage
    FROM bills
    GROUP BY bill_status
    ORDER BY bill_count DESC;
    """,
    conn
)

bill_status_rollup

,bill_status,bill_count,affected_accounts,total_bill_amount,avg_bill_amount,total_kwh_usage
0,Paid,39180,4682,28826168.08,735.74,1.183901e+08
1,Generated,11074,4361,8734387.43,788.73,3.591334e+07
2,Exception,5403,1673,0.00,NaN,1.282532e+07
3,Past Due,4343,2904,3291678.81,757.93,1.354608e+07


### Bill Status Rollup Findings

The bill status rollup provides a high-level summary of billing outcomes across the full synthetic billing dataset.

Key observations:

- Paid bills are the largest category, representing the majority of completed billing records.
- Generated bills represent bills that have been created but not yet paid or marked past due.
- Past-due bills create an accounts receivable monitoring use case for the dashboard.
- Exception bills have no calculated bill amount because they were intentionally blocked from normal billing due to missing reads, invalid reads, negative usage, or inactive account conditions.
- The rollup confirms that the billing dataset supports both financial reporting and operational exception monitoring.

This rollup is useful for dashboard KPI cards, while the detailed `vw_bill_status_summary` view supports drilldowns by billing cycle, county, customer segment, rate plan, and bill status.

## 10. Create Service Request Backlog View

This section creates a SQL view for service request backlog reporting. The view summarizes service requests by billing cycle, request type, priority, status, customer segment, and county.

This supports operational monitoring of open work, in-progress requests, closed requests, and request volume by workflow type.

In [13]:
# Create service request backlog view
create_service_request_backlog_view = """
DROP VIEW IF EXISTS vw_service_request_backlog;

CREATE VIEW vw_service_request_backlog AS
SELECT
    sr.billing_cycle_id,
    sr.customer_segment,
    sr.county,
    sr.request_type,
    sr.priority,
    sr.request_status,
    
    COUNT(DISTINCT sr.request_id) AS service_request_count,
    COUNT(DISTINCT sr.account_id) AS affected_accounts,
    
    SUM(CASE WHEN sr.request_status = 'Open' THEN 1 ELSE 0 END) AS open_requests,
    SUM(CASE WHEN sr.request_status = 'In Progress' THEN 1 ELSE 0 END) AS in_progress_requests,
    SUM(CASE WHEN sr.request_status = 'Closed' THEN 1 ELSE 0 END) AS closed_requests,
    SUM(CASE WHEN sr.request_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_requests,
    
    SUM(CASE WHEN sr.priority = 'Critical' THEN 1 ELSE 0 END) AS critical_requests,
    SUM(CASE WHEN sr.priority = 'High' THEN 1 ELSE 0 END) AS high_priority_requests,
    SUM(CASE WHEN sr.priority = 'Medium' THEN 1 ELSE 0 END) AS medium_priority_requests,
    SUM(CASE WHEN sr.priority = 'Low' THEN 1 ELSE 0 END) AS low_priority_requests

FROM service_requests sr
GROUP BY
    sr.billing_cycle_id,
    sr.customer_segment,
    sr.county,
    sr.request_type,
    sr.priority,
    sr.request_status;
"""

conn.executescript(create_service_request_backlog_view)

# Preview service request backlog view
service_request_backlog = pd.read_sql_query(
    """
    SELECT *
    FROM vw_service_request_backlog
    ORDER BY billing_cycle_id, service_request_count DESC
    LIMIT 20;
    """,
    conn
)

service_request_backlog

,billing_cycle_id,customer_segment,county,request_type,priority,request_status,service_request_count,affected_accounts,open_requests,in_progress_requests,closed_requests,cancelled_requests,critical_requests,high_priority_requests,medium_priority_requests,low_priority_requests
0,2024-01,Residential,CONTRA COSTA,Account Status Review,Low,Closed,5,5,0,0,5,0,0,0,0,5
1,2024-01,Residential,KERN,Account Status Review,Medium,Closed,5,5,0,0,5,0,0,0,5,0
2,2024-01,Residential,SANTA CLARA,Account Status Review,Medium,Closed,5,5,0,0,5,0,0,0,5,0
3,2024-01,Residential,ALAMEDA,Account Status Review,Medium,Closed,4,4,0,0,4,0,0,0,4,0
4,2024-01,Residential,CONTRA COSTA,Account Status Review,Medium,Closed,4,4,0,0,4,0,0,0,4,0
5,2024-01,Residential,KERN,Account Status Review,Medium,Open,4,4,4,0,0,0,0,0,4,0
6,2024-01,Residential,CONTRA COSTA,Account Status Review,Medium,Open,3,3,3,0,0,0,0,0,3,0
7,2024-01,Residential,KERN,Account Status Review,High,Closed,3,3,0,0,3,0,0,3,0,0
8,2024-01,Residential,SAN JOAQUIN,Account Status Review,Medium,Closed,3,3,0,0,3,0,0,0,3,0
9,2024-01,Residential,SANTA CLARA,Account Status Review,High,Closed,3,3,0,0,3,0,0,3,0,0


### Service Request Backlog View Findings

The `vw_service_request_backlog` view successfully summarizes service request activity by billing cycle, customer segment, county, request type, priority, and request status.

Key observations:

- The view supports detailed workflow reporting across open, in-progress, closed, and cancelled service requests.
- Account status review appears frequently because many service requests were generated from inactive-account billing exceptions.
- Missing read investigations and meter data investigations provide operational follow-up for meter-read-related billing exceptions.
- This detailed view is useful for Power BI drilldowns by month, geography, customer segment, request type, priority, and status.

The next step is to create a simpler service request rollup for notebook review and dashboard KPI cards.

In [14]:
# Create a simpler service request rollup for notebook review
service_request_rollup = pd.read_sql_query(
    """
    SELECT
        request_type,
        request_status,
        priority,
        COUNT(*) AS service_request_count,
        COUNT(DISTINCT account_id) AS affected_accounts,
        SUM(CASE WHEN request_status = 'Open' THEN 1 ELSE 0 END) AS open_requests,
        SUM(CASE WHEN request_status = 'In Progress' THEN 1 ELSE 0 END) AS in_progress_requests,
        SUM(CASE WHEN request_status = 'Closed' THEN 1 ELSE 0 END) AS closed_requests,
        SUM(CASE WHEN request_status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_requests
    FROM service_requests
    GROUP BY
        request_type,
        request_status,
        priority
    ORDER BY
        service_request_count DESC;
    """,
    conn
)

service_request_rollup.head(20)

,request_type,request_status,priority,service_request_count,affected_accounts,open_requests,in_progress_requests,closed_requests,cancelled_requests
0,Account Status Review,Closed,Medium,447,245,0,0,447,0
1,Account Status Review,Closed,Low,290,198,0,0,290,0
2,Account Status Review,Closed,High,201,153,0,0,201,0
3,Account Status Review,Open,Medium,155,118,155,0,0,0
4,Missing Read Investigation,Closed,Medium,127,125,0,0,127,0
5,Account Status Review,In Progress,Medium,123,98,0,123,0,0
6,Account Status Review,Open,Low,108,94,108,0,0,0
7,Account Status Review,In Progress,Low,90,81,0,90,0,0
8,Missing Read Investigation,Closed,Low,83,83,0,0,83,0
9,Account Status Review,Open,High,82,76,82,0,0,0


### Service Request Rollup Findings

The service request rollup provides a clearer summary of operational workflow volume by request type, status, and priority.

Key observations:

- Account status review is the largest service request category, which aligns with inactive-account billing exceptions being the largest exception type.
- Missing read investigations and meter data investigations provide operational follow-up for meter-read-related billing exceptions.
- Open and in-progress records create a useful backlog monitoring use case.
- Closed requests represent completed operational work, while cancelled requests represent requests that did not require completion.
- Priority levels support escalation reporting for high and critical requests.

This rollup is useful for dashboard KPI cards and backlog reporting, while the detailed `vw_service_request_backlog` view supports drilldowns by month, county, customer segment, request type, priority, and status.

## 11. Create Account Readiness View

This section creates a SQL view that evaluates account-level readiness for reporting, billing, and potential system migration. The view flags accounts with open billing exceptions, missing or invalid meter reads, inactive account status, and unresolved service requests.

This view supports data quality monitoring and migration-readiness analysis.

In [16]:
# Create account readiness view using pre-aggregated account-level summaries
create_account_readiness_view = """
DROP VIEW IF EXISTS vw_account_readiness;

CREATE VIEW vw_account_readiness AS
WITH meter_read_summary AS (
    SELECT
        account_id,
        COUNT(DISTINCT read_id) AS total_meter_reads,
        SUM(CASE WHEN read_status = 'Missing' THEN 1 ELSE 0 END) AS missing_meter_reads,
        SUM(CASE WHEN read_status = 'Invalid' THEN 1 ELSE 0 END) AS invalid_meter_reads
    FROM meter_reads
    GROUP BY account_id
),

bill_summary AS (
    SELECT
        account_id,
        COUNT(DISTINCT bill_id) AS total_bills,
        SUM(CASE WHEN bill_status = 'Exception' THEN 1 ELSE 0 END) AS exception_bills,
        SUM(CASE WHEN bill_status = 'Past Due' THEN 1 ELSE 0 END) AS past_due_bills
    FROM bills
    GROUP BY account_id
),

exception_summary AS (
    SELECT
        account_id,
        COUNT(DISTINCT exception_id) AS total_billing_exceptions,
        SUM(CASE WHEN resolution_status = 'Open' THEN 1 ELSE 0 END) AS open_billing_exceptions,
        SUM(CASE WHEN resolution_status = 'In Review' THEN 1 ELSE 0 END) AS in_review_billing_exceptions
    FROM billing_exceptions
    GROUP BY account_id
),

service_request_summary AS (
    SELECT
        account_id,
        COUNT(DISTINCT request_id) AS total_service_requests,
        SUM(CASE WHEN request_status = 'Open' THEN 1 ELSE 0 END) AS open_service_requests,
        SUM(CASE WHEN request_status = 'In Progress' THEN 1 ELSE 0 END) AS in_progress_service_requests
    FROM service_requests
    GROUP BY account_id
)

SELECT
    sa.account_id,
    sa.customer_id,
    sa.premise_id,
    sa.customer_segment,
    sa.county,
    sa.account_status,
    sa.rate_plan_id,
    m.meter_id,
    m.meter_type,
    m.meter_status,

    COALESCE(mrs.total_meter_reads, 0) AS total_meter_reads,
    COALESCE(mrs.missing_meter_reads, 0) AS missing_meter_reads,
    COALESCE(mrs.invalid_meter_reads, 0) AS invalid_meter_reads,

    COALESCE(bs.total_bills, 0) AS total_bills,
    COALESCE(bs.exception_bills, 0) AS exception_bills,
    COALESCE(bs.past_due_bills, 0) AS past_due_bills,

    COALESCE(es.total_billing_exceptions, 0) AS total_billing_exceptions,
    COALESCE(es.open_billing_exceptions, 0) AS open_billing_exceptions,
    COALESCE(es.in_review_billing_exceptions, 0) AS in_review_billing_exceptions,

    COALESCE(srs.total_service_requests, 0) AS total_service_requests,
    COALESCE(srs.open_service_requests, 0) AS open_service_requests,
    COALESCE(srs.in_progress_service_requests, 0) AS in_progress_service_requests,

    CASE
        WHEN sa.account_status != 'Active' THEN 0
        WHEN m.meter_status != 'Active' THEN 0
        WHEN COALESCE(mrs.missing_meter_reads, 0) > 0 THEN 0
        WHEN COALESCE(mrs.invalid_meter_reads, 0) > 0 THEN 0
        WHEN COALESCE(es.open_billing_exceptions, 0) > 0 THEN 0
        WHEN COALESCE(srs.open_service_requests, 0) + COALESCE(srs.in_progress_service_requests, 0) > 0 THEN 0
        ELSE 1
    END AS readiness_flag,

    CASE
        WHEN sa.account_status != 'Active' THEN 'Inactive Account'
        WHEN m.meter_status != 'Active' THEN 'Inactive Meter'
        WHEN COALESCE(mrs.missing_meter_reads, 0) > 0 THEN 'Missing Meter Reads'
        WHEN COALESCE(mrs.invalid_meter_reads, 0) > 0 THEN 'Invalid Meter Reads'
        WHEN COALESCE(es.open_billing_exceptions, 0) > 0 THEN 'Open Billing Exceptions'
        WHEN COALESCE(srs.open_service_requests, 0) + COALESCE(srs.in_progress_service_requests, 0) > 0 THEN 'Open Service Requests'
        ELSE 'Ready'
    END AS readiness_status

FROM service_accounts sa
LEFT JOIN meters m
    ON sa.account_id = m.account_id
LEFT JOIN meter_read_summary mrs
    ON sa.account_id = mrs.account_id
LEFT JOIN bill_summary bs
    ON sa.account_id = bs.account_id
LEFT JOIN exception_summary es
    ON sa.account_id = es.account_id
LEFT JOIN service_request_summary srs
    ON sa.account_id = srs.account_id;
"""

conn.executescript(create_account_readiness_view)

# Preview account readiness view
account_readiness = pd.read_sql_query(
    """
    SELECT *
    FROM vw_account_readiness
    ORDER BY readiness_flag ASC, total_billing_exceptions DESC
    LIMIT 20;
    """,
    conn
)

account_readiness

,account_id,customer_id,premise_id,customer_segment,county,account_status,rate_plan_id,meter_id,meter_type,meter_status,total_meter_reads,missing_meter_reads,invalid_meter_reads,total_bills,exception_bills,past_due_bills,total_billing_exceptions,open_billing_exceptions,in_review_billing_exceptions,total_service_requests,open_service_requests,in_progress_service_requests,readiness_flag,readiness_status
0,ACCT000032,CUST000032,PREM000032,Residential,PLACER,Pending Close,R-RES-TOU,MTR000032,Smart Meter,Inactive,12,0,1,12,12,0,12,5,1,7,1,2,0,Inactive Account
1,ACCT000050,CUST000050,PREM000050,Residential,PLACER,Pending Close,R-RES-TOU,MTR000050,Smart Meter,Removed,12,0,0,12,12,0,12,6,0,5,0,2,0,Inactive Account
2,ACCT000075,CUST000075,PREM000075,Residential,KINGS,Pending Close,R-RES-TOU,MTR000075,Smart Meter,Inactive,12,1,0,12,12,0,12,4,2,8,1,1,0,Inactive Account
3,ACCT000080,CUST000080,PREM000080,Residential,KERN,Inactive,R-RES-TOU,MTR000080,Smart Meter,Removed,12,0,0,12,12,0,12,3,7,7,1,0,0,Inactive Account
4,ACCT000101,CUST000101,PREM000101,Residential,KERN,Inactive,R-RES-TOU,MTR000101,Smart Meter,Inactive,12,0,1,12,12,0,12,4,1,7,1,2,0,Inactive Account
5,ACCT000103,CUST000103,PREM000103,Residential,SAN FRANCISCO,Pending Close,R-RES-TOU,MTR000103,Smart Meter,Inactive,12,0,0,12,12,0,12,5,1,8,0,1,0,Inactive Account
6,ACCT000131,CUST000131,PREM000131,Residential,SAN FRANCISCO,Pending Close,R-RES-TOU,MTR000131,Smart Meter,Inactive,12,0,0,12,12,0,12,4,1,5,1,2,0,Inactive Account
7,ACCT000211,CUST000211,PREM000211,Residential,NAPA,Inactive,R-RES-TOU,MTR000211,Smart Meter,Removed,12,0,0,12,12,0,12,2,2,5,0,2,0,Inactive Account
8,ACCT000223,CUST000223,PREM000223,Residential,SANTA CLARA,Inactive,R-RES-TOU,MTR000223,Smart Meter,Removed,12,0,0,12,12,0,12,6,1,10,2,1,0,Inactive Account
9,ACCT000246,CUST000246,PREM000246,Industrial,SAN FRANCISCO,Inactive,R-IND-GEN,MTR000246,Legacy Meter,Inactive,12,0,0,12,12,0,12,6,3,8,2,4,0,Inactive Account


In [17]:
# Summarize account readiness results
account_readiness_rollup = pd.read_sql_query(
    """
    SELECT
        readiness_status,
        readiness_flag,
        COUNT(*) AS account_count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM vw_account_readiness), 2) AS account_share_pct
    FROM vw_account_readiness
    GROUP BY
        readiness_status,
        readiness_flag
    ORDER BY
        readiness_flag ASC,
        account_count DESC;
    """,
    conn
)

account_readiness_rollup

,readiness_status,readiness_flag,account_count,account_share_pct
0,Missing Meter Reads,0,964,19.28
1,Invalid Meter Reads,0,391,7.82
2,Inactive Account,0,318,6.36
3,Open Service Requests,0,125,2.50
4,Ready,1,3202,64.04


### Account Readiness View Findings

The `vw_account_readiness` view evaluates each service account for billing and reporting readiness.

Key design choices:

- The view uses pre-aggregated account-level summaries for meter reads, bills, billing exceptions, and service requests.
- This avoids join fanout and keeps account-level counts reliable.
- Accounts are flagged as not ready if they have inactive account status, inactive meters, missing meter reads, invalid meter reads, open billing exceptions, or open/in-progress service requests.
- Accounts with no blocking issues are marked as ready.
- This view supports data quality monitoring, operational readiness reporting, and potential system migration-readiness analysis.

### Account Readiness Rollup Findings

The account readiness rollup provides a high-level view of which accounts are ready for reporting, billing operations, or potential system migration.

Key observations:

- 64.04% of accounts are marked as ready.
- Missing meter reads are the largest readiness blocker, affecting 19.28% of accounts.
- Invalid meter reads affect 7.82% of accounts and represent a data quality issue that should be resolved before normal billing or migration.
- Inactive account status affects 6.36% of accounts.
- Open service requests affect 2.50% of accounts.
- This view provides a practical data quality and migration-readiness layer for the project.

The readiness view is especially useful for enterprise application support because it identifies which accounts have blocking issues and why.

## 12. Create Monthly Billing Trend View

This section creates a SQL view for monthly billing trend analysis. The view summarizes bill volume, bill amount, kWh usage, and bill status counts by billing cycle.

This view is designed for time-series reporting in Power BI.

In [18]:
# Create monthly billing trend view
create_monthly_billing_trend_view = """
DROP VIEW IF EXISTS vw_monthly_billing_trend;

CREATE VIEW vw_monthly_billing_trend AS
SELECT
    b.billing_cycle_id,
    bc.cycle_start_date,
    bc.cycle_end_date,
    COUNT(DISTINCT b.bill_id) AS total_bills,
    COUNT(DISTINCT b.account_id) AS accounts_billed,

    ROUND(SUM(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount ELSE 0 END), 2) AS total_bill_amount,
    ROUND(AVG(CASE WHEN b.bill_amount IS NOT NULL THEN b.bill_amount END), 2) AS avg_bill_amount,
    ROUND(SUM(CASE WHEN b.kwh_usage >= 0 THEN b.kwh_usage ELSE 0 END), 2) AS total_kwh_usage,

    SUM(CASE WHEN b.bill_status = 'Paid' THEN 1 ELSE 0 END) AS paid_bills,
    SUM(CASE WHEN b.bill_status = 'Generated' THEN 1 ELSE 0 END) AS generated_bills,
    SUM(CASE WHEN b.bill_status = 'Past Due' THEN 1 ELSE 0 END) AS past_due_bills,
    SUM(CASE WHEN b.bill_status = 'Exception' THEN 1 ELSE 0 END) AS exception_bills,

    ROUND(
        100.0 * SUM(CASE WHEN b.bill_status = 'Exception' THEN 1 ELSE 0 END)
        / COUNT(DISTINCT b.bill_id),
        2
    ) AS exception_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN b.bill_status = 'Past Due' THEN 1 ELSE 0 END)
        / COUNT(DISTINCT b.bill_id),
        2
    ) AS past_due_rate_pct

FROM bills b
LEFT JOIN billing_cycles bc
    ON b.billing_cycle_id = bc.billing_cycle_id
GROUP BY
    b.billing_cycle_id,
    bc.cycle_start_date,
    bc.cycle_end_date;
"""

conn.executescript(create_monthly_billing_trend_view)

monthly_billing_trend = pd.read_sql_query(
    """
    SELECT *
    FROM vw_monthly_billing_trend
    ORDER BY billing_cycle_id;
    """,
    conn
)

monthly_billing_trend

,billing_cycle_id,cycle_start_date,cycle_end_date,total_bills,accounts_billed,total_bill_amount,avg_bill_amount,total_kwh_usage,paid_bills,generated_bills,past_due_bills,exception_bills,exception_rate_pct,past_due_rate_pct
0,2024-01,2024-01-01,2024-01-31,5000,5000,3486346.40,765.56,15364540.59,3271,888,395,446,8.92,7.90
1,2024-02,2024-02-01,2024-02-29,5000,5000,3019801.79,663.84,13477209.44,3284,890,375,451,9.02,7.50
2,2024-03,2024-03-01,2024-03-31,5000,5000,2967342.36,651.16,13355506.15,3289,902,366,443,8.86,7.32
3,2024-04,2024-04-01,2024-04-30,5000,5000,2800526.57,615.23,12678500.02,3256,906,390,448,8.96,7.80
4,2024-05,2024-05-01,2024-05-31,5000,5000,2849878.10,627.31,12908880.53,3298,895,350,457,9.14,7.00
5,2024-06,2024-06-01,2024-06-30,5000,5000,3177537.96,701.29,14223252.82,3243,948,340,469,9.38,6.80
6,2024-07,2024-07-01,2024-07-31,5000,5000,4432771.25,972.53,19095483.96,3223,971,364,442,8.84,7.28
7,2024-08,2024-08-01,2024-08-31,5000,5000,4376210.22,960.54,18920978.99,3234,975,347,444,8.88,6.94
8,2024-09,2024-09-01,2024-09-30,5000,5000,3671254.90,807.93,16116547.28,3196,976,372,456,9.12,7.44
9,2024-10,2024-10-01,2024-10-31,5000,5000,3820346.03,839.27,16721511.28,3329,891,332,448,8.96,6.64


### Monthly Billing Trend View Findings

The `vw_monthly_billing_trend` view summarizes monthly billing activity across the full synthetic Meter-to-Cash dataset.

Key observations:

- Each month contains 5,000 bills and 5,000 billed accounts, matching the synthetic billing design.
- Total bill amount and total kWh usage vary by month because meter reads were generated using real PG&E 2024 monthly usage multipliers.
- July and August show the highest total usage and bill amounts, reflecting the summer consumption peak from the real PG&E monthly consumption data.
- Lower billing totals appear in lower-demand months such as April, May, and November.
- The view includes paid, generated, past-due, and exception bill counts, making it useful for both financial and operational trend reporting.

This view is designed for Power BI line charts, KPI cards, and monthly billing performance dashboards.

## 13. Create PG&E Outage and Consumption Context View

This section creates a SQL view that summarizes PG&E county-level outage impact alongside 2024 county electricity consumption. The view provides real-data operational context for the synthetic Meter-to-Cash model.

In [19]:
# Create PG&E outage and consumption context view
create_pge_outage_consumption_context_view = """
DROP VIEW IF EXISTS vw_pge_outage_consumption_context;

CREATE VIEW vw_pge_outage_consumption_context AS
SELECT
    county,
    outage_incidents,
    total_impacted_customers,
    avg_impacted_customers,
    median_impacted_customers,
    avg_estimated_restoration_hours,
    max_estimated_restoration_hours,
    annual_total_gwh,
    annual_residential_gwh,
    annual_nonresidential_gwh,
    avg_monthly_gwh,
    impacted_customers_per_annual_gwh,
    incidents_per_annual_gwh
FROM pge_county_outage_consumption_context;
"""

conn.executescript(create_pge_outage_consumption_context_view)

pge_outage_consumption_context = pd.read_sql_query(
    """
    SELECT *
    FROM vw_pge_outage_consumption_context
    ORDER BY total_impacted_customers DESC
    LIMIT 20;
    """,
    conn
)

pge_outage_consumption_context

,county,outage_incidents,total_impacted_customers,avg_impacted_customers,median_impacted_customers,avg_estimated_restoration_hours,max_estimated_restoration_hours,annual_total_gwh,annual_residential_gwh,annual_nonresidential_gwh,avg_monthly_gwh,impacted_customers_per_annual_gwh,incidents_per_annual_gwh
0,CONTRA COSTA,6,4141,690.2,18.0,6.59,7.88,8468.06,3065.24,5402.82,705.67,0.4890,0.0007
1,GLENN,5,1458,291.6,319.0,NaN,NaN,423.20,97.84,325.36,35.27,3.4452,0.0118
2,SANTA CLARA,21,933,44.4,3.0,8.09,29.90,17793.58,4273.05,13520.54,1482.80,0.0524,0.0012
3,ALAMEDA,15,476,31.7,13.0,8.70,24.25,11699.70,3202.32,8497.38,974.97,0.0407,0.0013
4,SAN MATEO,16,434,27.1,9.5,6.23,10.05,4261.31,1614.59,2646.71,355.11,0.1018,0.0038
5,MARIN,3,327,109.0,1.0,8.60,10.22,1309.42,689.33,620.09,109.12,0.2497,0.0023
6,KERN,4,220,55.0,8.0,6.73,9.94,15272.88,2671.78,12601.10,1272.74,0.0144,0.0003
7,PLACER,2,202,101.0,101.0,6.81,7.89,3170.69,1689.26,1481.42,264.22,0.0637,0.0006
8,SAN FRANCISCO,6,100,16.7,1.0,6.37,9.47,5126.01,1501.42,3624.59,427.17,0.0195,0.0012
9,NEVADA,3,84,28.0,8.0,6.44,7.97,769.85,482.61,287.24,64.15,0.1091,0.0039


### PG&E Outage and Consumption Context View Findings

The `vw_pge_outage_consumption_context` view summarizes current PG&E outage impact alongside 2024 county electricity consumption.

Key observations:

- Contra Costa has the highest total impacted customer count in the current PG&E outage snapshot.
- Glenn has fewer total impacted customers than Contra Costa, but still appears as a high-impact county in the outage context.
- Santa Clara, Alameda, and San Mateo have higher outage incident counts, but their impact should be interpreted alongside county electricity consumption.
- Some counties have missing average or maximum estimated restoration values because not every outage incident included an estimated restoration timestamp.
- This view connects real PG&E outage activity to county-level demand context and can support Power BI visuals for outage impact, demand context, and normalized county comparison.

This view helps bridge the real utility operations data from Notebook 1 with the synthetic Meter-to-Cash data model created in Notebook 2.

## 14. Validate SQL Views Created

This section confirms that all analysis views were created successfully in the SQLite database.

In [20]:
# List all SQL views created in the database
sqlite_views = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'view'
    ORDER BY name;
    """,
    conn
)

sqlite_views

,name
0,vw_account_readiness
1,vw_bill_status_summary
2,vw_billing_exception_summary
3,vw_meter_to_cash_monthly_kpis
4,vw_monthly_billing_trend
5,vw_pge_outage_consumption_context
6,vw_service_request_backlog


### SQL View Creation Validation

The SQL view validation confirms that all planned analysis views were created successfully in the SQLite database.

The database now includes views for:

- Account readiness and data quality monitoring.
- Bill status reporting.
- Billing exception analysis.
- Monthly Meter-to-Cash KPIs.
- Monthly billing trends.
- PG&E outage and electricity consumption context.
- Service request backlog reporting.

These views create the reporting layer needed for SQL analysis, Power BI dashboard development, and project documentation.

## 15. Export SQL View Outputs for Reporting

This section exports the SQL analysis views as CSV files. These files can be used directly in Power BI or reviewed as reporting-ready outputs from the SQLite database.

In [21]:
# Define reporting output directory
REPORTING_DIR = Path("../data/reporting")
REPORTING_DIR.mkdir(parents=True, exist_ok=True)

# Export each SQL view to a reporting CSV
for view_name in sqlite_views["name"]:
    view_df = pd.read_sql_query(
        f"SELECT * FROM {view_name};",
        conn
    )
    
    view_df.to_csv(
        REPORTING_DIR / f"{view_name}.csv",
        index=False
    )

# Confirm exported reporting files
reporting_files = sorted([file.name for file in REPORTING_DIR.iterdir()])
reporting_files

['vw_account_readiness.csv',
 'vw_bill_status_summary.csv',
 'vw_billing_exception_summary.csv',
 'vw_meter_to_cash_monthly_kpis.csv',
 'vw_monthly_billing_trend.csv',
 'vw_pge_outage_consumption_context.csv',
 'vw_service_request_backlog.csv']

### Reporting Export Summary

The SQL analysis views were successfully exported as reporting-ready CSV files.

The exported files include:

- `vw_account_readiness.csv`
- `vw_bill_status_summary.csv`
- `vw_billing_exception_summary.csv`
- `vw_meter_to_cash_monthly_kpis.csv`
- `vw_monthly_billing_trend.csv`
- `vw_pge_outage_consumption_context.csv`
- `vw_service_request_backlog.csv`

These files can be used directly in Power BI for dashboard development or reviewed as standalone reporting outputs from the SQLite database.

## 16. Notebook Summary and Next Steps

This notebook created the SQL database and reporting view layer for the PG&E-style Utility Operations & Meter-to-Cash Analytics project.

Key outcomes:

- Loaded processed real-data tables from Notebook 1.
- Loaded synthetic Meter-to-Cash tables from Notebook 2.
- Created a SQLite database at `data/database/pge_meter_to_cash.db`.
- Wrote all processed and synthetic tables to SQLite.
- Validated SQLite table row counts against the source dataframe inventory.
- Validated key SQL relationships across customer, account, meter, billing, exception, and service request tables.
- Created SQL views for monthly Meter-to-Cash KPIs, bill status reporting, billing exception monitoring, service request backlog, account readiness, monthly billing trends, and PG&E outage/consumption context.
- Exported all SQL views as reporting-ready CSV files to `data/reporting/`.

The next phase of the project will focus on dashboard preparation and Power BI design. The reporting CSV files created in this notebook will serve as the main data sources for dashboard pages covering executive KPIs, billing performance, exceptions, service request backlog, account readiness, and PG&E outage/consumption context.